## Module 5 Class activities
This notebook is a starting point for the exercises and activities that we'll do in class. We'll do an extension of the random forests classifier, looking at a continuous variable.

Before you attempt any of these activities, make sure to watch the video lectures for this module.

### Classification: NYC evictions
We'll look at the factors that are associated with evictions in New York City. Perhaps a machine learning model can identify the types of places that are vulnerable to eviction, and target renter assistance programs more effectively?

#### Loading in the data

Let's start by loading in the [eviction dataset](https://data.cityofnewyork.us/City-Government/Evictions/6z8x-wfk4) via Socrata.

<div class="alert alert-block alert-info">

<strong>Exercise:</strong> Import the data from Socrata via the API into a pandas DataFrame.
</div>

*Hints*:
- Look back at Week 1 if you need a refresher on using Socrata
- There are about 70,000 rows in the dataset. So remember to add `?$limit=100000` to the end of the URL that you pass to `requests.get()`. Otherwise, you'll just get the first 1,000 rows. (The limit can be anything comfortably above 70000.)

In [1]:
import requests
import json
import pandas as pd
import geopandas as gpd

# your code here
url = 'https://data.cityofnewyork.us/resource/6z8x-wfk4.json?$limit=100000'
r = requests.get(url)
evictionDf = pd.DataFrame(json.loads(r.text)) 
evictionDf.head()

,court_index_number,docket_number,eviction_address,executed_date,marshal_first_name,marshal_last_name,residential_commercial_ind,borough,eviction_zip,ejectment,eviction_possession,latitude,longitude,community_board,council_district,census_tract,bin,bbl,nta,eviction_apt_num
0,71869/17,13576,80-14 ROOSEVELT AVENUE,2018-03-27T00:00:00.000,Edward,Guida,Commercial,QUEENS,11372,Not an Ejectment,Possession,40.747457,-73.885554,4,25,26901,4036844,4014910007,Elmhurst,NaN
1,70919/18,089709,110 ROCKAWAY PARKWAY,2019-04-12T00:00:00.000,Justin,Grossman,Residential,BROOKLYN,11212,Not an Ejectment,Possession,40.663737,-73.923051,17,41,892,3100108,3046150017,Brownsville,1R
2,065194/19,031861,300 WEST 46TH STREET,2023-08-10T00:00:00.000,Gary,Rose,Residential,MANHATTAN,10036,Not an Ejectment,Possession,40.759963,-73.988355,4,3,121,1081654,1010360036,Clinton,2-A1
3,79271/16,067040,287 LINDEN BOULEVARD,2017-01-09T00:00:00.000,Henry,Daley,Residential,BROOKLYN,11226,Not an Ejectment,Possession,40.652723,-73.948439,17,40,818,3108079,3048530090,Erasmus,6D
4,17773/18A,097702,245 WORTMAN AVENUE,2019-08-23T00:00:00.000,Justin,Grossman,Residential,BROOKLYN,11207,Not an Ejectment,Possession,40.657282,-73.884224,5,42,1106,3324016,3043710001,East New York,14J


<div class="alert alert-block alert-info">

<strong>Exercise:</strong> Convert your dataframe to a GeoDataFrame, using the latitude and longitude columns.

In [2]:
# your code here 

evictionGdf = gpd.GeoDataFrame(
    evictionDf, geometry=gpd.points_from_xy(evictionDf.longitude, evictionDf.latitude, 
                                          crs='EPSG:4326'))
evictionGdf

,court_index_number,docket_number,eviction_address,executed_date,marshal_first_name,marshal_last_name,residential_commercial_ind,borough,eviction_zip,ejectment,...,latitude,longitude,community_board,council_district,census_tract,bin,bbl,nta,eviction_apt_num,geometry
0,71869/17,13576,80-14 ROOSEVELT AVENUE,2018-03-27T00:00:00.000,Edward,Guida,Commercial,QUEENS,11372,Not an Ejectment,...,40.747457,-73.885554,4,25,26901,4036844,4014910007,Elmhurst,NaN,POINT (-73.88555 40.74746)
1,70919/18,089709,110 ROCKAWAY PARKWAY,2019-04-12T00:00:00.000,Justin,Grossman,Residential,BROOKLYN,11212,Not an Ejectment,...,40.663737,-73.923051,17,41,892,3100108,3046150017,Brownsville,1R,POINT (-73.92305 40.66374)
2,065194/19,031861,300 WEST 46TH STREET,2023-08-10T00:00:00.000,Gary,Rose,Residential,MANHATTAN,10036,Not an Ejectment,...,40.759963,-73.988355,4,3,121,1081654,1010360036,Clinton,2-A1,POINT (-73.98836 40.75996)
3,79271/16,067040,287 LINDEN BOULEVARD,2017-01-09T00:00:00.000,Henry,Daley,Residential,BROOKLYN,11226,Not an Ejectment,...,40.652723,-73.948439,17,40,818,3108079,3048530090,Erasmus,6D,POINT (-73.94844 40.65272)
4,17773/18A,097702,245 WORTMAN AVENUE,2019-08-23T00:00:00.000,Justin,Grossman,Residential,BROOKLYN,11207,Not an Ejectment,...,40.657282,-73.884224,5,42,1106,3324016,3043710001,East New York,14J,POINT (-73.88422 40.65728)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,64508/19-2,028724,104-36 196TH STREET,2019-11-15T00:00:00.000,Gary,Rose,Residential,QUEENS,11412,Not an Ejectment,...,40.707738,-73.762224,12,27,504,4232979,4108910021,Hollis,1B,POINT (-73.76222 40.70774)
99996,73299/19B,109195,510 WEST END AVENUE,2022-08-24T00:00:00.000,Justin,Grossman,Residential,MANHATTAN,10024,Not an Ejectment,...,40.788002,-73.978488,7,6,171,1033125,1012320063,Upper West Side,10,POINT (-73.97849 40.788)
99997,77918/16,9022,360 EAST 55TH STREET,2017-07-12T00:00:00.000,Edward,Guida,Residential,MANHATTAN,10022,Not an Ejectment,...,40.757423,-73.964875,6,5,108,1039951,1013470030,Turtle Bay-East Midtown,12F,POINT (-73.96488 40.75742)
99998,K92009/18,112560,170 33RD STREET,2019-03-21T00:00:00.000,Darlene,Barone,Residential,BROOKLYN,11232,Not an Ejectment,...,40.655718,-74.002245,7,38,84,3010179,3006850008,Sunset Park West,3,POINT (-74.00224 40.65572)


Now let's import some census data. We could use `cenpy` or the Census Bureau API. But to keep things simple so that we can focus on the spatial joins and the machine learning, I downloaded the block group-level 2019 ACS data for New York from the [Census Bureau](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-data.html). To save space, I clipped it to the 5 NYC counties.

It's in your repository, and we can load it in as follows. If you aren't familiar with a GeoPackage (GPKG) format, think of it as a "new and improved shapefile." [Here's a good overview.](https://towardsdatascience.com/why-you-need-to-use-geopackage-files-instead-of-shapefile-or-geojson-7cb24fe56416)

In [3]:
bgs = gpd.read_file('data/nyc_bgs.gpkg')
bgs.head()

,GEOID,B01001e1,B01001e10,B01001e11,B01001e12,B01001e13,B01001e14,B01001e15,B01001e16,B01001e17,...,B19001e8,B19001e9,B22010e1,B22010e2,B22010e3,B22010e4,B22010e5,B22010e6,B22010e7,geometry
0,15000US360050175002,656,39,0,0,0,18,14,0,22,...,11,0,358,214,139,75,144,107,37,"POLYGON ((-73.9157 40.83054, -73.91485 40.8302..."
1,15000US360050141001,1228,0,35,96,26,45,28,0,54,...,34,0,503,291,226,65,212,70,142,"POLYGON ((-73.91661 40.82499, -73.91592 40.825..."
2,15000US360050145001,2716,44,192,33,38,76,30,64,93,...,137,83,972,534,316,218,438,7,431,"POLYGON ((-73.90584 40.83106, -73.90505 40.832..."
3,15000US360050075002,3488,43,109,122,169,19,51,139,22,...,63,37,1188,470,300,170,718,147,571,"POLYGON ((-73.91035 40.81995, -73.91022 40.820..."
4,15000US360050418001,657,0,38,5,11,21,14,43,25,...,0,0,217,87,18,69,130,41,89,"POLYGON ((-73.86288 40.89515, -73.86146 40.897..."


Note that the variables aren't particularly carefully selected - I just threw in many of the demographic and housing variables. 

Nor are the variable names particularly informative, but the full names are in a file in the repository.

In [4]:
# note it is tab-sepated, not comma separated
# so we use the sep='\t' argument

col_names = pd.read_csv('data/BG_METADATA_2019.txt', sep='\t', index_col='Short_Name')
col_names.head()

,Full_Name
Short_Name,
B01001e1,SEX BY AGE: Total: Total population -- (Estimate)
B01001m1,SEX BY AGE: Total: Total population -- (Margin...
B01001e2,SEX BY AGE: Male: Total population -- (Estimate)
B01001m2,SEX BY AGE: Male: Total population -- (Margin ...
B01001e3,SEX BY AGE: Male: Under 5 years: Total populat...


So you can see the definition of the column like this. (I don't recommend renaming the `bg` column names, because the full names are so long.)

In [5]:
col_names.loc['B01001e1']

Full_Name    SEX BY AGE: Total: Total population -- (Estimate)
Name: B01001e1, dtype: object

#### Spatial join
Now let's do the spatial join. Again, let's follow our three step process.

1. Use a spatial join to add the `GEOID` column to the evictions dataframe. *Hint:* Check your projections.
2. Group by `GEOID` to get a count of evictions per block group. If you have a `Series`, give it a name - maybe `n_evictions`
3. Join those counts back - a tabular join based on the index

<div class="alert alert-block alert-info">
    <strong>Exercise:</strong> Add a count of evictions per census block group to your <strong>bgs</strong> GeoDataFrame, using the 3-step process above.
</div>

In [6]:
print(evictionGdf.crs)
print(bgs.crs)

EPSG:4326
EPSG:4269


In [7]:
evictionGdf_sjoin = gpd.sjoin(evictionGdf, bgs.to_crs('EPSG:4326'), predicate='intersects')
print(len(evictionGdf))
print(len(bgs))
print(len(evictionGdf_sjoin))

100000
6493
91066


In [8]:
evictionGdf_sjoin.groupby('GEOID').size()

GEOID
15000US360050002001    10
15000US360050002002    18
15000US360050002003     9
15000US360050004001    15
15000US360050004002    19
                       ..
15000US360850319012    12
15000US360850319021    78
15000US360850319022    41
15000US360850319023    34
15000US360850323001    21
Length: 5930, dtype: int64

In [9]:
temp_evictionDf = evictionGdf_sjoin.groupby('GEOID').size()
temp_evictionDf = pd.DataFrame(temp_evictionDf)
temp_evictionDf.columns = ['n_evictions']

In [10]:
temp_evictionDf

,n_evictions
GEOID,
15000US360050002001,10
15000US360050002002,18
15000US360050002003,9
15000US360050004001,15
15000US360050004002,19
...,...
15000US360850319012,12
15000US360850319021,78
15000US360850319022,41


In [11]:
evictionGdf_sjoin.set_index('GEOID', inplace=True) 

In [12]:
evictionGdf_sjoin

,court_index_number,docket_number,eviction_address,executed_date,marshal_first_name,marshal_last_name,residential_commercial_ind,borough,eviction_zip,ejectment,...,B19001e7,B19001e8,B19001e9,B22010e1,B22010e2,B22010e3,B22010e4,B22010e5,B22010e6,B22010e7
GEOID,,,,,,,,,,,,,,,,,,,,,
15000US360810269011,71869/17,13576,80-14 ROOSEVELT AVENUE,2018-03-27T00:00:00.000,Edward,Guida,Commercial,QUEENS,11372,Not an Ejectment,...,45,62,20,597,109,64,45,488,95,393
15000US360470892002,70919/18,089709,110 ROCKAWAY PARKWAY,2019-04-12T00:00:00.000,Justin,Grossman,Residential,BROOKLYN,11212,Not an Ejectment,...,23,14,68,531,174,77,97,357,87,270
15000US360610121005,065194/19,031861,300 WEST 46TH STREET,2023-08-10T00:00:00.000,Gary,Rose,Residential,MANHATTAN,10036,Not an Ejectment,...,0,0,0,488,88,59,29,400,22,378
15000US360470818001,79271/16,067040,287 LINDEN BOULEVARD,2017-01-09T00:00:00.000,Henry,Daley,Residential,BROOKLYN,11226,Not an Ejectment,...,21,40,0,1160,155,51,104,1005,51,954
15000US360471106001,17773/18A,097702,245 WORTMAN AVENUE,2019-08-23T00:00:00.000,Justin,Grossman,Residential,BROOKLYN,11207,Not an Ejectment,...,0,64,42,838,478,279,199,360,113,247
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15000US360810504001,64508/19-2,028724,104-36 196TH STREET,2019-11-15T00:00:00.000,Gary,Rose,Residential,QUEENS,11412,Not an Ejectment,...,11,32,0,280,50,50,0,230,39,191
15000US360610171003,73299/19B,109195,510 WEST END AVENUE,2022-08-24T00:00:00.000,Justin,Grossman,Residential,MANHATTAN,10024,Not an Ejectment,...,0,0,27,651,166,120,46,485,28,457
15000US360610108005,77918/16,9022,360 EAST 55TH STREET,2017-07-12T00:00:00.000,Edward,Guida,Residential,MANHATTAN,10022,Not an Ejectment,...,26,0,0,697,26,0,26,671,25,646


In [13]:
bgs = bgs.set_index('GEOID').join(temp_evictionDf)
bgs.fillna({'n_evictions':0}, inplace=True)
bgs.head()

,B01001e1,B01001e10,B01001e11,B01001e12,B01001e13,B01001e14,B01001e15,B01001e16,B01001e17,B01001e18,...,B19001e9,B22010e1,B22010e2,B22010e3,B22010e4,B22010e5,B22010e6,B22010e7,geometry,n_evictions
GEOID,,,,,,,,,,,,,,,,,,,,,
15000US360050175002,656,39,0,0,0,18,14,0,22,0,...,0,358,214,139,75,144,107,37,"POLYGON ((-73.9157 40.83054, -73.91485 40.8302...",33.0
15000US360050141001,1228,0,35,96,26,45,28,0,54,0,...,0,503,291,226,65,212,70,142,"POLYGON ((-73.91661 40.82499, -73.91592 40.825...",57.0
15000US360050145001,2716,44,192,33,38,76,30,64,93,71,...,83,972,534,316,218,438,7,431,"POLYGON ((-73.90584 40.83106, -73.90505 40.832...",48.0
15000US360050075002,3488,43,109,122,169,19,51,139,22,31,...,37,1188,470,300,170,718,147,571,"POLYGON ((-73.91035 40.81995, -73.91022 40.820...",111.0
15000US360050418001,657,0,38,5,11,21,14,43,25,2,...,0,217,87,18,69,130,41,89,"POLYGON ((-73.86288 40.89515, -73.86146 40.897...",27.0


In [14]:
# your code here



<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Do a quick-and-dirty map of the number of evictions. This will help identify any data holes.
</div>

In [15]:
# your code here

#### Random forests regressor
Now we have our data set. Let's estimate a random forests model.

In contrast to the examples in the lecture, we are trying to predict a continuous variable - the number of evictions. So our classifier isn't appropriate. 

However, there is a similar model: the [random forest regressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor). It works almost identically to the classifier. The main difference from a user perspective is assessing model performance - a confusion matrix doesn't work here.

You'll need to follow the following steps:
- choose your x variables. (Your y variable will be `n_evictions`)
- Drop Null values if needed
- split your dataset into training and testing portions
- estimate (fit) the model

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Estimate a random forest regressor model to predict the number of evictions per census tract.</div>

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# your code here

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Examine some of your trees in the random forest. What do they tell you?</div>

In [17]:
# your code here

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Experiment with different model hyperparameters and variables. Discuss your rationale and the results with a neighbor.</div>

In [18]:
# your code here

The following questions relate to some of the material in Module 6. You might want to wait until watching those lectures. Then come back and complete these tasks.

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Assess the fit of your model.</div>

Remember, the confusion matrix and accuracy scores don't apply to continuous data. Some ideas for continuous variables are [here](https://stackoverflow.com/questions/50789508/random-forest-regression-how-do-i-analyse-its-performance-python-sklearn). You could also plot actual vs predicted values.

In [19]:
# your code here

<div class="alert alert-block alert-info">
<strong>Exercise:</strong> Which variables are most important in your predictions? Plot the forest importances.</div>

In [20]:
# your code here


<div class="alert alert-block alert-info">
<h3>What you should have learned</h3>
<ul>
  <li>Get more practice with spatial joins and Socrata.</li>
  <li>Learn how to estimate a random forests model for continuous data.</li>
</ul>
</div>